# Deutsch-Jozsa Algorithm

Use a single oracle query to decide whether a hidden function is constant or balanced -- a task that costs a classical computer up to $2^{n-1}+1$ queries.

**Objectives:**
- Build constant and balanced oracles for $n=3$ input qubits plus one ancilla
- Understand phase kickback: an ancilla in |-> turns $f(x)$ into a sign $(-1)^{f(x)}$
- Run the superpose -> query -> interfere pattern and read the verdict from the input register
- Prove with the exact state vector that constant gives all-zeros with certainty and balanced never does
- Compare the single quantum query to the classical worst case

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt
from lib.utils.statevector import statevector

# Use local simulator by default (free, instant)
device = LocalSimulator()

## 1. The setup: constant vs. balanced

You are handed a function $f:\{0,1\}^3 \to \{0,1\}$ with a promise: it is either

- **constant** -- the same output for every input (all 0s, or all 1s), or
- **balanced** -- output 0 on exactly half the inputs and 1 on the other half.

Classically, to be *certain* which case you have, you may need to check just over half the
inputs: $2^{n-1}+1$ queries (here $2^2+1 = 5$). Deutsch-Jozsa decides it with **one** query.

We use $n=3$ input qubits (qubit indices 0, 1, 2) plus **1 ancilla** (qubit 3), for 4 qubits
total. The oracle $U_f$ is a reversible gate that computes $|x\rangle|y\rangle \mapsto
|x\rangle|y \oplus f(x)\rangle$.

Remember qcsim is **big-endian**: qubit 0 is the leftmost bit of every measured bitstring. So in
a 4-character outcome, the **input register is `bitstring[:3]`** and the **ancilla is
`bitstring[3]`**.

In [ ]:
# Register layout
N_INPUT = 3      # input qubits: 0, 1, 2
ANCILLA = 3      # single ancilla qubit
N_TOTAL = 4      # total qubits

print(f"Input register: qubits 0..{N_INPUT - 1}  ->  bitstring[:{N_INPUT}]")
print(f"Ancilla: qubit {ANCILLA}  ->  bitstring[{ANCILLA}]")
print(f"Classical worst case to be certain: 2^(n-1)+1 = {2**(N_INPUT - 1) + 1} queries")

## 2. The three oracles

Each oracle is a small Braket `Circuit` acting on the 4 qubits. We will compose it into the full
algorithm with `add_circuit`.

- **Constant-0**: $f(x)=0$ for all $x$. It does nothing to the ancilla -- the identity. We place
  an `i` (identity) gate on each qubit so the circuit explicitly spans all 4 wires.
- **Constant-1**: $f(x)=1$ for all $x$. It flips the ancilla unconditionally: a single `x` on the
  ancilla.
- **Balanced (parity)**: $f(x) = x_0 \oplus x_1 \oplus x_2$. A `cnot` from **each** input qubit
  onto the ancilla XORs all three input bits into it. This is 0 for half the inputs and 1 for the
  other half -- balanced by construction.

In [ ]:
def oracle_constant_0():
    """f(x) = 0 for all x. Identity on the ancilla (does nothing)."""
    c = Circuit()
    for q in range(N_TOTAL):
        c.i(q)  # identity on every wire so the circuit spans all 4 qubits
    return c


def oracle_constant_1():
    """f(x) = 1 for all x. Flip the ancilla unconditionally."""
    c = Circuit()
    c.x(ANCILLA)
    return c


def oracle_balanced_parity():
    """f(x) = x0 XOR x1 XOR x2. CNOT each input qubit onto the ancilla."""
    c = Circuit()
    for q in range(N_INPUT):
        c.cnot(q, ANCILLA)
    return c


print("Balanced (parity) oracle:")
print(oracle_balanced_parity())

## 3. Assembling Deutsch-Jozsa

The recipe is the oracle-algorithm pattern from the GUIDE -- *superpose, query once, interfere*:

1. Put the ancilla into |-> : apply `x` then `h` on qubit 3. (|-> = (|0> - |1>)/sqrt(2).)
2. Apply `h` to the three input qubits, creating an equal superposition over all 8 inputs.
3. Query the oracle **once**.
4. Apply `h` to the three input qubits again to interfere.

**Phase kickback** is the crux. With the ancilla in |->, the oracle's action
$|y\rangle \mapsto |y \oplus f(x)\rangle$ becomes a *sign*:

$$
|x\rangle\,|-\rangle \;\longmapsto\; (-1)^{f(x)}\,|x\rangle\,|-\rangle .
$$

The ancilla is left untouched; the function value rides home as a phase on the input register. The
final amplitude on $|000\rangle$ is $\tfrac{1}{8}\sum_x (-1)^{f(x)}$: it is $\pm 1$ when $f$ is
constant (every term agrees) and exactly $0$ when $f$ is balanced (the $+$ and $-$ terms cancel).

In [ ]:
def deutsch_jozsa(oracle):
    """Full DJ circuit for a given oracle sub-circuit."""
    c = Circuit()
    # Step 1: ancilla into |-> (X then H)
    c.x(ANCILLA)
    # Step 1-2: H on all 4 qubits (ancilla already X'd, inputs go to superposition)
    for q in range(N_TOTAL):
        c.h(q)
    # Step 3: query the oracle once
    c.add_circuit(oracle)
    # Step 4: H on the input register to interfere
    for q in range(N_INPUT):
        c.h(q)
    return c


dj = deutsch_jozsa(oracle_balanced_parity())
print("Deutsch-Jozsa with the balanced (parity) oracle:")
print(dj)
print(f"\nQubits: {dj.qubit_count}, depth: {dj.depth}")

## 4. The exact verdict from the state vector

No sampling needed for the proof: `statevector(circuit)` gives the exact final amplitudes. We sum
$|\text{amplitude}|^2$ over every basis state whose **top 3 bits are `000`** -- that is the
probability the input register reads all-zeros. Because qcsim is big-endian, a basis index $i$
formatted as a 4-bit string has the input register in positions `[:3]` and the ancilla in `[3]`.

- Constant oracle  =>  all-zeros probability **= 1** (the verdict "constant").
- Balanced oracle  =>  all-zeros probability **= 0** (so we measure something nonzero => "balanced").

In [ ]:
def input_register_zero_prob(circuit):
    """Exact P(input register == 000), summed from the state vector."""
    sv = statevector(circuit)
    n = circuit.qubit_count
    total = 0.0
    for idx in range(len(sv)):
        bits = format(idx, f"0{n}b")  # big-endian: bits[:3] is the input register
        if bits[:N_INPUT] == "0" * N_INPUT:
            total += abs(sv[idx]) ** 2
    return float(total)


p_const0 = input_register_zero_prob(deutsch_jozsa(oracle_constant_0()))
p_const1 = input_register_zero_prob(deutsch_jozsa(oracle_constant_1()))
p_bal = input_register_zero_prob(deutsch_jozsa(oracle_balanced_parity()))

print(f"constant-0 : P(input == 000) = {p_const0:.6f}")
print(f"constant-1 : P(input == 000) = {p_const1:.6f}")
print(f"balanced   : P(input == 000) = {p_bal:.6f}")

# Exact asserts (state vector, not sampling): constant => all-zeros with certainty.
assert np.isclose(p_const0, 1.0), "constant-0 must leave the input register in |000>"
assert np.isclose(p_const1, 1.0), "constant-1 must leave the input register in |000>"
# Balanced => the all-zeros amplitude cancels exactly to zero.
assert np.isclose(p_bal, 0.0), "balanced must have zero amplitude on the all-zeros input"
print("\nProved: constant -> P(000)=1 with certainty; balanced -> P(000)=0 exactly.")

## 5. A second balanced oracle

Balanced does not have to mean "parity of all bits". Any $f$ that is 0 on half the inputs works.
A single `cnot` from qubit 0 to the ancilla implements $f(x)=x_0$, which is 0 for the four inputs
with $x_0=0$ and 1 for the four with $x_0=1$ -- balanced. It must also give all-zeros probability 0.

In [ ]:
def oracle_balanced_q0():
    """f(x) = x0. Balanced: 0 on half the inputs, 1 on the other half."""
    c = Circuit()
    c.cnot(0, ANCILLA)
    return c


p_bal2 = input_register_zero_prob(deutsch_jozsa(oracle_balanced_q0()))
print(f"balanced (f = x0): P(input == 000) = {p_bal2:.6f}")
assert np.isclose(p_bal2, 0.0), "a balanced oracle must never leave the input in |000>"
print("Confirmed: a different balanced oracle also gives P(000) = 0.")

## 6. Now measure it: one shot decides

The state vector is the proof; sampling is the demonstration. We run each circuit on the
simulator, then **aggregate the 4-qubit outcomes by the input-register substring** `bitstring[:3]`
(ignoring the ancilla). The decision rule:

- input register **all-zeros**  =>  the function is **constant**
- input register **anything else**  =>  the function is **balanced**

In [ ]:
from collections import Counter


def run_and_aggregate(circuit, shots=2000):
    """Run the circuit and aggregate counts by the input-register substring."""
    result = device.run(circuit, shots=shots).result()
    agg = Counter()
    for bitstring, count in result.measurement_counts.items():
        agg[bitstring[:N_INPUT]] += count  # top 3 bits = input register
    return agg


def verdict(agg, shots):
    """Constant iff all shots land on the all-zeros input register."""
    zeros = "0" * N_INPUT
    return "constant" if agg.get(zeros, 0) == shots else "balanced"


SHOTS = 2000
labelled = {
    "constant-0": oracle_constant_0(),
    "constant-1": oracle_constant_1(),
    "balanced-parity": oracle_balanced_parity(),
    "balanced-x0": oracle_balanced_q0(),
}

for name, orc in labelled.items():
    agg = run_and_aggregate(deutsch_jozsa(orc), shots=SHOTS)
    print(f"{name:18s} input-register counts={dict(agg)}  ->  {verdict(agg, SHOTS)}")

## 7. Visualizing the interference

A histogram of the input-register outcomes makes the verdict obvious. The constant oracle piles
every shot onto `000`; the balanced oracle puts **zero** weight on `000` and spreads the rest
elsewhere. (For the parity oracle the weight lands entirely on `111`, the bit-string the oracle
encodes -- exactly the Bernstein-Vazirani readout mentioned in the GUIDE.)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
all_inputs = [format(i, f"0{N_INPUT}b") for i in range(2 ** N_INPUT)]

for ax, (name, orc) in zip(axes, [("constant-1", oracle_constant_1()),
                                  ("balanced-parity", oracle_balanced_parity())]):
    agg = run_and_aggregate(deutsch_jozsa(orc), shots=SHOTS)
    heights = [agg.get(b, 0) for b in all_inputs]
    ax.bar(all_inputs, heights, color="#4c72b0")
    ax.set_title(f"{name}: input-register counts")
    ax.set_xlabel("input register (bitstring[:3])")
    ax.set_ylabel("counts")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
print("Left: all weight on 000 (constant). Right: zero weight on 000 (balanced).")

## 8. One query vs. the classical worst case

The quantum circuit calls the oracle **once**, regardless of $n$. A classical algorithm has no
phase to interfere; in the worst case it must evaluate $f$ on $2^{n-1}+1$ distinct inputs before
it can rule out "constant". For $n=3$ that is 5 queries -- and the gap grows exponentially with
$n$.

In [ ]:
classical_worst_case = 2 ** (N_INPUT - 1) + 1
quantum_queries = 1

print(f"n = {N_INPUT} input qubits")
print(f"Classical worst case: {classical_worst_case} oracle queries")
print(f"Quantum (Deutsch-Jozsa): {quantum_queries} oracle query")

# The DJ circuit contains exactly one oracle invocation by construction.
assert quantum_queries == 1
assert classical_worst_case == 5
print("\nDeutsch-Jozsa decides in a single query what classically needs up to 5.")

## Exercises

Two exercises to deepen the intuition. Each has tiered hints -- expand only
what you need -- and a check cell that confirms the property once you have
it. Use the helpers above (`deutsch_jozsa`, `input_register_zero_prob`,
`run_and_aggregate`) and keep everything on the local simulator.

### Exercise 1 — A third balanced oracle

Section 5 built one alternative balanced oracle, $f(x)=x_0$. Build another:
$f(x) = x_1 \oplus x_2$. Then push it through the same pipeline and confirm it
really is balanced.

Define `oracle_balanced_x1x2` -- a function that returns the oracle `Circuit` --
and `p_x1x2`, the exact all-zeros input-register probability of the full
Deutsch-Jozsa circuit built from it. For a balanced function that probability
must come out 0.

<details><summary>Hint 1 — nudge</summary>

The parity oracle in Section 2 XORed *all three* input bits into the ancilla.
This function only involves two of them. Which two qubits carry $x_1$ and $x_2$,
and which one gets left out?

</details>
<details><summary>Hint 2 — approach</summary>

Inside the function start a fresh `Circuit()` and route a `cnot` from qubit 1
onto the ancilla and another from qubit 2 onto the ancilla -- the same
one-CNOT-per-variable recipe as `oracle_balanced_parity`, just without qubit 0.
Then reuse the helpers: feed `deutsch_jozsa(oracle_balanced_x1x2())` into
`input_register_zero_prob(...)` to get `p_x1x2`.

</details>

In [ ]:
# Exercise 1: Build a third balanced oracle, f(x) = x1 XOR x2.
# Define: oracle_balanced_x1x2 (a function returning the oracle Circuit) and
#         p_x1x2, the exact all-zeros input-register probability of its DJ circuit.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    _orc = oracle_balanced_x1x2()
    assert _orc.qubit_count <= N_TOTAL, "the oracle should stay within the 4-qubit register"
    assert 0.0 <= p_x1x2 <= 1.0, "a probability must live in [0, 1]"
    assert np.isclose(p_x1x2, 0.0), (
        "a balanced oracle leaves zero amplitude on the all-zeros input register"
    )
    _agg = run_and_aggregate(deutsch_jozsa(oracle_balanced_x1x2()), shots=2000)
    assert _agg.get("000", 0) == 0, (
        "a balanced verdict means the input register never reads all-zeros"
    )

### Exercise 2 — Watch the phase kickback

The claim behind the whole algorithm is that the oracle leaves the ancilla
untouched and only stamps a sign on the input register. Look at it directly:
build just steps 1-3 (ancilla into |->, `h` on the three inputs, then the parity
oracle) and STOP before the final interfering `h` layer.

Define `sv_kickback` as `statevector(...)` of that partial circuit. If the
ancilla is still in |->, then for every input pattern the two amplitudes that
differ only in the ancilla bit are equal in magnitude and opposite in sign.

<details><summary>Hint 1 — nudge</summary>

Section 6 wrote the post-oracle state as $(-1)^{f(x)}\,|x\rangle\,|-\rangle$;
the |-> factor never changed. Since $|-\rangle = (|0\rangle - |1\rangle)/\sqrt2$,
what relationship must hold between the two amplitudes for a fixed input $x$ with
the ancilla at 0 versus at 1?

</details>
<details><summary>Hint 2 — approach</summary>

Take the first three steps of `deutsch_jozsa` and drop the closing input-H loop:
`x` on the ancilla, `h` on all four qubits, then
`add_circuit(oracle_balanced_parity())`. Call `statevector(...)` on that circuit.
Because qcsim is big-endian the ancilla is the last bit, so an even index has
the ancilla at 0 and the very next (odd) index is the same input with the
ancilla at 1.

</details>

In [ ]:
# Exercise 2: Inspect the phase kickback before the final H layer.
# Define: sv_kickback -- statevector after steps 1-3 (ancilla -> |->, H on the
#         inputs, then the parity oracle), WITHOUT the closing input-H layer.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert len(sv_kickback) == 2 ** N_TOTAL, (
        "the state vector should span all 4 qubits (16 amplitudes)"
    )
    _a0 = [sv_kickback[2 * x] for x in range(2 ** N_INPUT)]      # ancilla bit = 0
    _a1 = [sv_kickback[2 * x + 1] for x in range(2 ** N_INPUT)]  # ancilla bit = 1
    assert all(np.isclose(abs(z0), abs(z1)) for z0, z1 in zip(_a0, _a1)), (
        "the ancilla-0 and ancilla-1 amplitudes should match in magnitude"
    )
    assert all(np.isclose(z1, -z0) for z0, z1 in zip(_a0, _a1)), (
        "still in |->: the two ancilla amplitudes are equal and opposite for every input"
    )
    _signs = {int(np.sign(z0.real)) for z0 in _a0 if abs(z0) > 1e-9}
    assert _signs == {-1, 1}, (
        "a balanced oracle should stamp BOTH + and - signs across the input register"
    )

## Summary

- Deutsch-Jozsa decides **constant vs. balanced** with a **single** oracle query; the classical
  worst case is $2^{n-1}+1$ (5 for $n=3$).
- The engine is **phase kickback**: an ancilla in |-> turns the oracle's bit-flip into a sign
  $(-1)^{f(x)}$ on the input register, leaving the ancilla unchanged.
- The final $H^{\otimes n}$ interferes those signs. The all-zeros amplitude is
  $\tfrac{1}{N}\sum_x (-1)^{f(x)}$: $\pm 1$ for constant, exactly $0$ for balanced.
- Because qcsim is big-endian and measures all qubits at once, you read the verdict from the
  **input-register substring** `bitstring[:3]`; all-zeros means constant.
- We proved both claims exactly from `statevector(...)` (no flaky sampling) and demonstrated the
  one-shot decision on the simulator.

**Next:** [`02-grovers-search.ipynb`](02-grovers-search.ipynb) -- when one query is not enough, and
interference must be *amplified* step by step.